In [3]:
##统计分类准确率和F1值
import json
from sklearn.metrics import accuracy_score, f1_score

def calculate_classification_metrics(results_file, output_file):
    """计算分类准确率和F1值"""
    print(f"Loading results from {results_file}...")
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    # 收集所有预测和标签
    all_predictions = []
    all_labels = []
    
    for item in results:
        if item['predicted_label'] is not None and item['label'] is not None:
            all_predictions.append(item['predicted_label'])
            all_labels.append(item['label'])
    
    # 计算总体指标
    if len(all_predictions) > 0:
        # 计算准确率
        accuracy = accuracy_score(all_labels, all_predictions)
        
        metrics_results = {
            'accuracy': round(accuracy, 4),
            'num_samples': len(all_predictions),
            'num_correct': sum(1 for p, l in zip(all_predictions, all_labels) if p == l)
        }
    else:
        metrics_results = {
            'accuracy': None,
            'num_samples': 0,
            'num_correct': 0
        }
    
    # 保存结果
    print(f"Saving classification metrics to {output_file}...")
    with open(output_file, 'w') as f:
        json.dump(metrics_results, f, indent=2, ensure_ascii=False)
    
    # 打印结果
    print("\n" + "="*60)
    print("Classification Metrics:")
    print("="*60)
    print(f"Accuracy:      {metrics_results['accuracy']:.4f}")
    print(f"Correct/Total: {metrics_results['num_correct']}/{metrics_results['num_samples']}")
    print("="*60)
    
    return metrics_results

results_file = '/path/to/inference_results.json'
output_file = '/path/to/classification_metrics.json'
calculate_classification_metrics(results_file, output_file)

Loading results from Causal_RL/eval/AW_FB/only_caugrpo/results.json...
Saving classification metrics to Causal_RL/eval/AW_FB/only_caugrpo/classification_metrics.json...

Classification Metrics:
Accuracy:      0.1607
Correct/Total: 90/560


{'accuracy': 0.1607, 'num_samples': 560, 'num_correct': 90}

In [23]:
##统计推理质量得分
import json
from collections import defaultdict
def analyze_by_subtask(eval_file, output_file=None):
    """
    
    参数:
        eval_file: 评测结果JSON文件路径
        output_file: 输出统计结果的JSON文件路径（可选）
    """
    print(f"Loading evaluation results from {eval_file}...")
    with open(eval_file, 'r') as f:
        results = json.load(f)
    
    print(f"Total samples: {len(results)}")

    
    dimensions = ['readability', 'logical_consistency', 'comprehensiveness']
    
    # 计算平均分
    summary = {}
    
    # 计算总体平均分
    overall_stats = {
        'readability': [],
        'logical_consistency': [],
        'comprehensiveness': []
    }
    
    for item in results:
        evaluations = item.get('evaluations', {})
        for dimension in dimensions:
            if dimension in evaluations:
                score = evaluations[dimension].get('score')
                if score is not None:
                    overall_stats[dimension].append(score)
    
    summary['Overall'] = {}
    for dimension in dimensions:
        scores = overall_stats[dimension]
        if scores:
            avg_score = sum(scores) / len(scores)
            summary['Overall'][dimension] = {
                'average': round(avg_score, 3),
                'num_valid': len(scores)
            }
        else:
            summary['Overall'][dimension] = {
                'average': None,
                'num_valid': 0
            }
    
    summary['Overall']['count'] = len(results)
    
    # 保存结果
    if output_file:
        print(f"\nSaving summary to {output_file}...")
        with open(output_file, 'w') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
    
    # 打印总体统计
    if 'Overall' in summary:
        stats = summary['Overall']
        count = stats['count']
        
        read_avg = stats['readability']['average']
        read_str = f"{read_avg:.3f}" if read_avg is not None else "N/A"
        
        logic_avg = stats['logical_consistency']['average']
        logic_str = f"{logic_avg:.3f}" if logic_avg is not None else "N/A"
        
        comp_avg = stats['comprehensiveness']['average']
        comp_str = f"{comp_avg:.3f}" if comp_avg is not None else "N/A"
        
        print(f"{'Overall':<15} {count:<8} {read_str:<15} {logic_str:<15} {comp_str:<15}")
    
    print("="*80)
    
    return summary



eval_file = '/path/to/eval_results.json'
output_file = '/path/to/eval_summary.json'
    
analyze_by_subtask(eval_file, output_file)

Loading evaluation results from Causal_RL/eval/AW_FB/causal_grpo/0321_kimik2_eval_results.json...
Total samples: 560

Saving summary to Causal_RL/eval/AW_FB/causal_grpo/kimik2_eval_summary.json...
Overall         560      3.000           2.798           2.996          


{'Overall': {'readability': {'average': 3.0, 'num_valid': 560},
  'logical_consistency': {'average': 2.798, 'num_valid': 560},
  'comprehensiveness': {'average': 2.996, 'num_valid': 560},
  'count': 560}}